In [139]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [140]:
pd.set_option('display.max_columns', None)
df.head()

,Sales,Quantity,Discount,Shipping Days,High Discount,Value per Item,Category,SubCategory,Region,Segment,ShipMode,Discount Bucket,Returned
0,261.9600,2,0.00,3,0,130.979935,Furniture,Bookcases,South,Consumer,Second Class,0-20%,0
1,731.9400,3,0.00,3,0,243.979919,Furniture,Chairs,South,Consumer,Second Class,0-20%,0
2,14.6200,2,0.00,4,0,7.309996,Office Supplies,Labels,West,Corporate,Second Class,0-20%,0
3,957.5775,5,0.45,7,1,191.515462,Furniture,Tables,South,Consumer,Standard Class,40-60%,0
4,22.3680,2,0.20,7,0,11.183994,Office Supplies,Storage,South,Consumer,Standard Class,0-20%,0


In [141]:
curr_dir = os.getcwd()
root = os.path.abspath(os.path.join(curr_dir,'..'))
raw_data_path = os.path.join(root,'data','raw')
processed_data_path = os.path.join(root,'data','processed')
file_name = "cleaned_df.csv"

In [142]:
try:
    orders = pd.read_csv(os.path.join(processed_data_path,file_name),parse_dates =["OrderDate", "ShipDate"])
    print("Data loaded successfully")


except FileNotFoundError as e:
    print(f"File not found: {e}")

except Exception as e:
    print(f"An error occurred: {e}")



Data loaded successfully


In [143]:
orders.head(2)

,OrderId,OrderDate,ShipDate,ShipMode,CustomerId,CustomerName,Segment,Country,City,State,PostalCode,Region,ProductId,Category,SubCategory,ProductName,Sales,Quantity,Discount,Profit,Shipping Days,Returned
0,CA-2017-152156,2017-11-08,2017-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0.0,41.9136,3,0
1,CA-2017-152156,2017-11-08,2017-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.94,3,0.0,219.5820,3,0


### Feature Engineering

In [144]:
# Profit Margin
orders["Profit Margin"] = orders["Profit"] / (orders["Sales"] + 1e-6)

# Discount Bucket
bins = [-0.01, 0.2, 0.4, 0.6, 0.8, 1.0]
labels = ["0-20%", "20-40%", "40-60%", "60-80%", "80-100%"]

orders["Discount Bucket"] = pd.cut(orders["Discount"], bins=bins, labels=labels)

# High Discount Flag
orders["High Discount"] = (orders["Discount"] > 0.2).astype(int)

# Order Value per Item
orders["Value per Item"] = orders["Sales"] / (orders["Quantity"] + 1e-6)

orders.head()

,OrderId,OrderDate,ShipDate,ShipMode,CustomerId,CustomerName,Segment,Country,City,State,PostalCode,Region,ProductId,Category,SubCategory,ProductName,Sales,Quantity,Discount,Profit,Shipping Days,Returned,Profit Margin,Discount Bucket,High Discount,Value per Item
0,CA-2017-152156,2017-11-08,2017-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136,3,0,0.1600,0-20%,0,130.979935
1,CA-2017-152156,2017-11-08,2017-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820,3,0,0.3000,0-20%,0,243.979919
2,CA-2017-138688,2017-06-12,2017-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714,4,0,0.4700,0-20%,0,7.309996
3,US-2016-108966,2016-10-11,2016-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310,7,0,-0.4000,40-60%,1,191.515462
4,US-2016-108966,2016-10-11,2016-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164,7,0,0.1125,0-20%,0,11.183994


In [145]:
orders.columns

Index(['OrderId', 'OrderDate', 'ShipDate', 'ShipMode', 'CustomerId',
       'CustomerName', 'Segment', 'Country', 'City', 'State', 'PostalCode',
       'Region', 'ProductId', 'Category', 'SubCategory', 'ProductName',
       'Sales', 'Quantity', 'Discount', 'Profit', 'Shipping Days', 'Returned',
       'Profit Margin', 'Discount Bucket', 'High Discount', 'Value per Item'],
      dtype='object')

In [146]:
features = [
    "Sales",
    "Quantity",
    "Discount",
    "Shipping Days",
    "High Discount",
    "Value per Item",
    "Category",
    "SubCategory",
    "Region",
    "Segment",
    "ShipMode",
    "Discount Bucket"
]

target = "Returned"

df = orders[features + [target]].copy()

In [147]:
df.isnull().sum()

Sales              0
Quantity           0
Discount           0
Shipping Days      0
High Discount      0
Value per Item     0
Category           0
SubCategory        0
Region             0
Segment            0
ShipMode           0
Discount Bucket    0
Returned           0
dtype: int64

### Encoding

#### One-hot Encoding

In [148]:
df_encoded = pd.get_dummies(df, drop_first=True)

In [149]:
df_encoded = df_encoded.astype(int)

In [150]:
df_encoded

,Sales,Quantity,Discount,Shipping Days,High Discount,Value per Item,Returned,Category_Office Supplies,Category_Technology,SubCategory_Appliances,SubCategory_Art,SubCategory_Binders,SubCategory_Bookcases,SubCategory_Chairs,SubCategory_Copiers,SubCategory_Envelopes,SubCategory_Fasteners,SubCategory_Furnishings,SubCategory_Labels,SubCategory_Machines,SubCategory_Paper,SubCategory_Phones,SubCategory_Storage,SubCategory_Supplies,SubCategory_Tables,Region_East,Region_South,Region_West,Segment_Corporate,Segment_Home Office,ShipMode_Same Day,ShipMode_Second Class,ShipMode_Standard Class,Discount Bucket_20-40%,Discount Bucket_40-60%,Discount Bucket_60-80%,Discount Bucket_80-100%
0,261,2,0,3,0,130,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0
1,731,3,0,3,0,243,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0
2,14,2,0,4,0,7,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,1,0,0,1,0,0,0,0,0
3,957,5,0,7,1,191,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,1,0,1,0,0
4,22,2,0,7,0,11,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9977,25,3,0,2,0,8,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0
9978,91,2,0,5,0,45,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0
9979,258,2,0,5,0,129,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0
9980,29,4,0,5,0,7,1,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0


In [151]:
from sklearn.model_selection import train_test_split

X = df_encoded.drop(columns=[target])
y = df_encoded[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [152]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("Before:", y_train.value_counts())
print("After:", y_train_smote.value_counts())

Before: Returned
0    7345
1     640
Name: count, dtype: int64
After: Returned
1    7345
0    7345
Name: count, dtype: int64


c:\Users\Administrator\OneDrive\Desktop\projects\FUTURE_DS_01\venv\lib\site-packages\sklearn\base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


In [153]:
print(X_train.shape)
print(X_train_smote.shape)

(7985, 36)
(14690, 36)


In [154]:
X_train_smote

,Sales,Quantity,Discount,Shipping Days,High Discount,Value per Item,Category_Office Supplies,Category_Technology,SubCategory_Appliances,SubCategory_Art,SubCategory_Binders,SubCategory_Bookcases,SubCategory_Chairs,SubCategory_Copiers,SubCategory_Envelopes,SubCategory_Fasteners,SubCategory_Furnishings,SubCategory_Labels,SubCategory_Machines,SubCategory_Paper,SubCategory_Phones,SubCategory_Storage,SubCategory_Supplies,SubCategory_Tables,Region_East,Region_South,Region_West,Segment_Corporate,Segment_Home Office,ShipMode_Same Day,ShipMode_Second Class,ShipMode_Standard Class,Discount Bucket_20-40%,Discount Bucket_40-60%,Discount Bucket_60-80%,Discount Bucket_80-100%
0,170,3,0,5,0,56,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0
1,60,3,0,6,0,20,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0
2,75,6,0,4,0,12,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
3,2,3,0,4,1,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,1,0,0,1,0
4,14,1,0,4,0,14,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14685,313,6,0,2,0,44,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0
14686,39,1,0,3,0,39,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0
14687,45,4,0,5,0,11,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
14688,119,1,0,6,0,119,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0


In [155]:
y_train_smote


0        1
1        0
2        0
3        0
4        0
        ..
14685    1
14686    1
14687    1
14688    1
14689    1
Name: Returned, Length: 14690, dtype: int64

In [156]:
print("Profit" in X_train_smote.columns)
print("Profit Margin" in X_train_smote.columns)

False
False


In [159]:
print(y_train_smote.value_counts())

Returned
1    7345
0    7345
Name: count, dtype: int64


### MODEL TRAINING

#### Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

lr = LogisticRegression(max_iter=1000, class_weight='balanced')

lr.fit(X_train_smote, y_train_smote)

y_pred_lr = lr.predict_proba(X_test)[:,1]


print("Logistic Regression")
print(confusion_matrix(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))




c:\Users\Administrator\OneDrive\Desktop\projects\FUTURE_DS_01\venv\lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Logistic Regression


ValueError: Classification metrics can't handle a mix of binary and continuous targets

In [167]:
lr = LogisticRegression(
    max_iter=2000,
    class_weight='balanced'
)

lr.fit(X_train_smote, y_train_smote)

# Probability
y_proba_lr = lr.predict_proba(X_test)[:, 1]

# Threshold tuning
threshold = 0.3
y_pred_lr = (y_proba_lr > threshold).astype(int)

print("🔹 Logistic Regression (Threshold = 0.3)")
print(confusion_matrix(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))

🔹 Logistic Regression (Threshold = 0.3)
[[1401  436]
 [ 106   54]]
              precision    recall  f1-score   support

           0       0.93      0.76      0.84      1837
           1       0.11      0.34      0.17       160

    accuracy                           0.73      1997
   macro avg       0.52      0.55      0.50      1997
weighted avg       0.86      0.73      0.78      1997



c:\Users\Administrator\OneDrive\Desktop\projects\FUTURE_DS_01\venv\lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


#### Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42)

rf.fit(X_train_smote, y_train_smote)

y_pred_rf = rf.predict(X_test)

print("Random Forest")
print(confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

Random Forest
[[1752   85]
 [ 149   11]]
              precision    recall  f1-score   support

           0       0.92      0.95      0.94      1837
           1       0.11      0.07      0.09       160

    accuracy                           0.88      1997
   macro avg       0.52      0.51      0.51      1997
weighted avg       0.86      0.88      0.87      1997



In [168]:
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    class_weight='balanced',
    random_state=42
)

rf.fit(X_train_smote, y_train_smote)

# Probability
y_proba_rf = rf.predict_proba(X_test)[:, 1]

# Threshold tuning
threshold = 0.3
y_pred_rf = (y_proba_rf > threshold).astype(int)

print("🌲 Random Forest (Threshold = 0.3)")
print(confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

🌲 Random Forest (Threshold = 0.3)
[[1111  726]
 [  65   95]]
              precision    recall  f1-score   support

           0       0.94      0.60      0.74      1837
           1       0.12      0.59      0.19       160

    accuracy                           0.60      1997
   macro avg       0.53      0.60      0.47      1997
weighted avg       0.88      0.60      0.69      1997



### XG Boost

In [ ]:
xgb = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=1,  # already balanced with SMOTE
    random_state=42
)

xgb.fit(X_train_smote, y_train_smote)

y_pred_xgb = xgb.predict(X_test)

print("XGBoost")
print(confusion_matrix(y_test, y_pred_xgb))
print(classification_report(y_test, y_pred_xgb))

XGBoost
[[1712  125]
 [ 149   11]]
              precision    recall  f1-score   support

           0       0.92      0.93      0.93      1837
           1       0.08      0.07      0.07       160

    accuracy                           0.86      1997
   macro avg       0.50      0.50      0.50      1997
weighted avg       0.85      0.86      0.86      1997



In [169]:
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=3,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)

xgb.fit(X_train_smote, y_train_smote)

# Probability
y_proba_xgb = xgb.predict_proba(X_test)[:, 1]

c:\Users\Administrator\OneDrive\Desktop\projects\FUTURE_DS_01\venv\lib\site-packages\xgboost\core.py:158: UserWarning: [17:36:41] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


In [171]:
thresholds = [0.5, 0.3, 0.2]

for t in thresholds:
    print(f"\n XGBoost Results (Threshold = {t})")
    
    y_pred = (y_proba_xgb > t).astype(int)
    
    print(confusion_matrix(y_test, y_pred))
    print(classification_report(y_test, y_pred))


 XGBoost Results (Threshold = 0.5)
[[1624  213]
 [ 137   23]]
              precision    recall  f1-score   support

           0       0.92      0.88      0.90      1837
           1       0.10      0.14      0.12       160

    accuracy                           0.82      1997
   macro avg       0.51      0.51      0.51      1997
weighted avg       0.86      0.82      0.84      1997


 XGBoost Results (Threshold = 0.3)
[[1413  424]
 [ 115   45]]
              precision    recall  f1-score   support

           0       0.92      0.77      0.84      1837
           1       0.10      0.28      0.14       160

    accuracy                           0.73      1997
   macro avg       0.51      0.53      0.49      1997
weighted avg       0.86      0.73      0.78      1997


 XGBoost Results (Threshold = 0.2)
[[1225  612]
 [ 100   60]]
              precision    recall  f1-score   support

           0       0.92      0.67      0.77      1837
           1       0.09      0.38      0.14     